# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shrinidhi-02/flyrank-machine-learning/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Answer

My lane is a scoring task. The goal is to assign each page a content refresh opportunity score so that pages can be prioritized for human review. A higher score would indicate that a page may deserve more attention based on its available content and search-performance signals. The score would support prioritization rather than automatically deciding that a page must be refreshed.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Answer

The target is a content refresh opportunity score for each page. Because the starter dataset does not contain an observed outcome showing whether a page actually benefited from a refresh, I would use a proxy based on available signals such as content age, impressions, average position, and trend direction. This proxy would be used to prioritize pages for human review, not as proof that a refresh will improve performance.

In [ ]:
# Sketch the target/proxy column using the available trend label.
lane_df["refresh_proxy"] = (
    lane_df["trend_direction"] == "down"
).astype(int)

display(
    lane_df[
        ["trend_direction", "refresh_proxy"]
    ].head(10)
)

print("refresh_proxy = 1 means the page is declining.")
print("refresh_proxy = 0 means the page is not marked as declining.")

,trend_direction,refresh_proxy
0,down,1
1,down,1
2,down,1
3,stable,0
4,down,1
5,down,1
6,down,1
7,stable,0
8,down,1
9,down,1


refresh_proxy = 1 means the page is declining.
refresh_proxy = 0 means the page is not marked as declining.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Answer

I will use Precision@50 as the success metric. It measures how many of the top 50 pages prioritized by the scoring system are actually declining pages according to the available data. A higher Precision@50 means the prioritized list contains more pages that may deserve human review, making the scoring system more useful for content refresh prioritization.

In [ ]:
# Provisional priority score for demonstrating the metric.
# Higher content age and worse average position give higher priority.

lane_df["priority_score"] = (
    lane_df["content_age_days"].rank(pct=True)
    + lane_df["avg_position"].rank(pct=True)
)

top_50 = lane_df.nlargest(50, "priority_score")

precision_at_50 = top_50["refresh_proxy"].mean()

print(f"Precision@50: {precision_at_50:.3f}")
print(f"Declining pages in top 50: {top_50['refresh_proxy'].sum()}")
print("A higher Precision@50 means the top-priority list contains more declining pages.")

Precision@50: 0.240
Declining pages in top 50: 12
A higher Precision@50 means the top-priority list contains more declining pages.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/Shrinidhi-02/flyrank-machine-learning/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

lane_df = df[
    (df["content_age_days"] >= 90) &
    (df["impressions_90d"] > 0)
].copy()

display(
    lane_df[
        [
            "content_age_days",
            "impressions_90d",
            "avg_position",
            "trend_direction",
            "trend_pct"
        ]
    ].head(10)
)

print(f"Number of pages in the lane: {len(lane_df):,}")
print("Unit of analysis: one row represents one webpage.")

,content_age_days,impressions_90d,avg_position,trend_direction,trend_pct
0,187,3803,10.6,down,-41.4
1,445,15320,20.3,down,-57.7
2,141,12581,36.5,down,-60.9
3,463,11751,6.2,stable,-13.8
4,263,19140,44.0,down,-34.7
5,147,3970,8.5,down,-38.9
6,90,20,7.0,down,-92.3
7,445,1724,21.2,stable,0.6
8,90,32574,46.0,down,-58.8
9,257,1240,4.9,down,-29.2


Number of pages in the lane: 30,000
Unit of analysis: one row represents one webpage.


In [2]:
cols = [
    "trend_direction",
    "trend_pct",
    "content_age_days",
    "impressions_90d",
    "avg_position"
]

print("Total rows:", len(df))

print("\n1. trend_direction value counts")
print(df["trend_direction"].value_counts(dropna=False))

print("\n2. trend_pct describe()")
print(df["trend_pct"].describe())

print("\n3. trend_pct by trend_direction")
print(df.groupby("trend_direction", dropna=False)["trend_pct"].describe())

print("\n4. Numeric columns describe()")
print(df[["content_age_days", "impressions_90d", "avg_position"]].describe())

age_ok = df["content_age_days"] >= 90
impr_ok = df["impressions_90d"] > 0
both_ok = age_ok & impr_ok

print("\n5. Row counts after filters")
print("content_age_days >= 90:", int(age_ok.sum()))
print("impressions_90d > 0:", int(impr_ok.sum()))
print("Both conditions:", int(both_ok.sum()), "of", len(df))
print("Rows removed:", len(df) - int(both_ok.sum()))

print("\n6. Missing values")
print(df[cols].isna().sum())

Total rows: 30000

1. trend_direction value counts
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

2. trend_pct describe()
count    26612.000000
mean        -4.785969
std        473.861780
min       -100.000000
25%        -62.600000
50%        -33.500000
75%          0.000000
max      44900.000000
Name: trend_pct, dtype: float64

3. trend_pct by trend_direction
                   count        mean          std    min   25%    50%    75%  \
trend_direction                                                                
down             16262.0  -58.113830    23.488605 -100.0 -75.9 -55.60  -38.5   
flat                 0.0         NaN          NaN    NaN   NaN    NaN    NaN   
new                  0.0         NaN          NaN    NaN   NaN    NaN    NaN   
stable            5962.0   -3.185944    11.054723  -20.0 -12.7  -3.80    5.0   
up                4388.0  190.673997  1145.029477   20.0  36.5  62.55  123.1   



In [3]:
print("Missing trend_pct by trend_direction")
print(
    df[df["trend_pct"].isna()]
    ["trend_direction"]
    .value_counts()
)

Missing trend_pct by trend_direction
trend_direction
new     2236
flat    1152
Name: count, dtype: int64


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Answer

A fixed rule could prioritize pages using simple conditions such as content age or a declining trend, but this may miss different combinations of signals across pages. Machine Learning can learn patterns from multiple available signals and produce a score that helps prioritize pages more flexibly. This is useful when the relationship between page age, impressions, position, and performance is not captured well by one fixed threshold. The output would still support human review rather than automatically deciding which pages should be refreshed.

In [ ]:
print("ML framing check:")
print(f"Pages available for analysis: {len(lane_df):,}")
print("Signals available:", [
    "content_age_days",
    "impressions_90d",
    "avg_position",
    "trend_direction"
])
print("The scoring output is intended for human review and prioritization.")

ML framing check:
Pages available for analysis: 30,000
Signals available: ['content_age_days', 'impressions_90d', 'avg_position', 'trend_direction']
The scoring output is intended for human review and prioritization.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.